# ETL Pipeline — Data Cleaning & Transformation

**Mục tiêu của file này:**
1. **Extract**: Đọc 9 file dữ liệu thô từ `data/raw/`.
2. **Clean & Transform**:
   - Chuẩn hóa mã sản phẩm, xử lý khoảng trắng, hợp nhất đầy đủ catalog từ các layout kho.
   - Điền giá trị thiếu (Impute missing `Size (US)`).
   - Tách cấu trúc vị trí kho (Zone, Aisle, Slot), tính khoảng cách Euclidean từ cửa xuất, hợp nhất 100% vị trí trong kho.
   - Parse chuỗi ngày tháng `creationDate`.
   - Trích xuất tọa độ điểm điều hướng.
   - **Unpivot/Melt** 18 cột slot trong 4 file chiến lược kho (`Random`, `Class_Based`, `Dedicated`, `Hybrid`) thành dạng bảng chuẩn hóa.
3. **Load to Staging**: Lưu toàn bộ các bảng sạch vào thư mục `data/staging/`.

> **Lưu ý:** File này **KHÔNG** tạo bảng DIM, FACT. Việc tạo Star Schema và nạp Database sẽ được thực hiện riêng biệt tại `ETL/DataWH.ipynb`.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Cấu hình đường dẫn ──────────────────────────────────────────
BASE_DIR = Path(r'd:\Cong Viec\Tailieu261\DataWarehouse\BTL')
RAW_DIR  = BASE_DIR / 'data' / 'raw'
STG_DIR  = BASE_DIR / 'data' / 'staging'

STG_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup hoàn tất!")
print(f"   RAW_DIR : {RAW_DIR}")
print(f"   STG_DIR : {STG_DIR}")

✅ Setup hoàn tất!
   RAW_DIR : d:\Cong Viec\Tailieu261\DataWarehouse\BTL\data\raw
   STG_DIR : d:\Cong Viec\Tailieu261\DataWarehouse\BTL\data\staging


---
## 1. EXTRACT — Đọc Dữ Liệu Thô (Raw Data)

In [2]:
print("📂 Đang đọc 9 file raw CSV...")

# Đọc đúng separator và encoding đã phát hiện ở bước EDA
raw_product    = pd.read_csv(RAW_DIR / 'Product.csv', sep=';', encoding='utf-8-sig')
raw_order      = pd.read_csv(RAW_DIR / 'Customer_Order.csv', sep=';', encoding='utf-8-sig')
raw_wave       = pd.read_csv(RAW_DIR / 'Picking_Wave.csv', sep=';', encoding='utf-8-sig')
raw_location   = pd.read_csv(RAW_DIR / 'Storage_Location.csv')
raw_nav        = pd.read_csv(RAW_DIR / 'Support_Points_Navigation.csv', sep=';', encoding='utf-8-sig')

raw_random     = pd.read_csv(RAW_DIR / 'Random_Storage.csv')
raw_classbased = pd.read_csv(RAW_DIR / 'Class_Based_Storage.csv', sep=';', encoding='utf-8-sig')
raw_dedicated  = pd.read_csv(RAW_DIR / 'Dedicated_Storage.csv', sep=';', encoding='utf-8-sig')
raw_hybrid     = pd.read_csv(RAW_DIR / 'Hybrid_Storage.csv', sep=';', encoding='utf-8-sig')

print("✅ Đã load thành công 9 files!")
print(f"   Product                  : {raw_product.shape}")
print(f"   Customer_Order           : {raw_order.shape}")
print(f"   Picking_Wave             : {raw_wave.shape}")
print(f"   Storage_Location         : {raw_location.shape}")
print(f"   Support_Points_Navigation: {raw_nav.shape}")
print(f"   Random_Storage           : {raw_random.shape}")
print(f"   Class_Based_Storage      : {raw_classbased.shape}")
print(f"   Dedicated_Storage        : {raw_dedicated.shape}")
print(f"   Hybrid_Storage           : {raw_hybrid.shape}")

📂 Đang đọc 9 file raw CSV...
✅ Đã load thành công 9 files!
   Product                  : (208, 3)
   Customer_Order           : (122370, 9)
   Picking_Wave             : (215192, 6)
   Storage_Location         : (2292, 5)
   Support_Points_Navigation: (44, 2)
   Random_Storage           : (2292, 19)
   Class_Based_Storage      : (2341, 20)
   Dedicated_Storage        : (2340, 20)
   Hybrid_Storage           : (2340, 20)


---
## 2. TRANSFORM & CLEANING Từng Bảng

In [3]:
# ── 2.1 Làm sạch Product (Bao gồm cả các SKU bổ sung từ layout kho) ──
print("🔄 [1/6] Làm sạch Product...")
df_stg_product = raw_product.copy()

df_stg_product['Reference'] = df_stg_product['Reference'].str.strip()
df_stg_product['ABCCOD']    = df_stg_product['ABCCOD'].str.strip().str.upper()
df_stg_product['Sector']    = df_stg_product['Sector'].str.strip()

# Trích xuất các SKU xuất hiện trong Class_Based_Storage nhưng chưa có trong Product.csv
cbs_cols = [c for c in raw_classbased.columns if c not in ['Location', 'ABCCOD']]
cbs_melt = raw_classbased.melt(id_vars=['Location', 'ABCCOD'], value_vars=cbs_cols)
cbs_prods_series = cbs_melt['value'].dropna().astype(str).str.split(';').str[0].str.strip()
cbs_sku_abc = pd.DataFrame({'ref': cbs_prods_series, 'abc': cbs_melt['ABCCOD']}).drop_duplicates('ref').set_index('ref')['abc'].to_dict()

known_skus = set(df_stg_product['Reference'])
extra_skus = sorted(list(set(cbs_prods_series) - known_skus))

if extra_skus:
    print(f"   Bổ sung {len(extra_skus)} SKU từ layout kho vào danh mục sản phẩm...")
    extra_prod_rows = []
    for sku in extra_skus:
        extra_prod_rows.append({
            'Reference': sku,
            'ABCCOD'   : cbs_sku_abc.get(sku, 'C'),
            'Sector'   : 'PF'
        })
    df_stg_product = pd.concat([df_stg_product, pd.DataFrame(extra_prod_rows)], ignore_index=True)

df_stg_product = df_stg_product.rename(columns={
    'Reference': 'reference',
    'ABCCOD'   : 'abc_code',
    'Sector'   : 'sector'
})

print(f"   Hoàn tất: {len(df_stg_product)} sản phẩm trong danh mục.")
display(df_stg_product.head(3))

🔄 [1/6] Làm sạch Product...
   Bổ sung 109 SKU từ layout kho vào danh mục sản phẩm...
   Hoàn tất: 317 sản phẩm trong danh mục.


,reference,abc_code,sector
0,O9YFO8,A,PF
1,I1X92B,A,PF
2,HOUGRO,B,PF


In [4]:
# ── 2.2 Làm sạch Storage_Location & Hợp nhất toàn bộ vị trí kho ──────
print("🔄 [2/6] Làm sạch Storage_Location...")
df_stg_location = raw_location.copy()

df_stg_location['originalLocation'] = df_stg_location['originalLocation'].str.strip()
df_stg_location['position']         = df_stg_location['position'].str.strip()

# Tách Zone, Aisle, Slot (Định dạng chuẩn: Zone-Aisle-Slot)
loc_parsed = df_stg_location['originalLocation'].str.extract(r'([A-Za-z]+)-(\d+)-(\d+)')
df_stg_location['zone']  = loc_parsed[0]
df_stg_location['aisle'] = pd.to_numeric(loc_parsed[1], errors='coerce')
df_stg_location['slot']  = pd.to_numeric(loc_parsed[2], errors='coerce')

# Tính khoảng cách Euclidean từ gốc (cửa xuất kho: 0, 0, 0)
df_stg_location['dist_from_origin'] = np.sqrt(
    df_stg_location['x']**2 + df_stg_location['y']**2 + (df_stg_location['z'] * 10)**2
).round(2)

df_stg_location = df_stg_location.rename(columns={
    'originalLocation': 'location_code',
    'position'        : 'position_str'
})[['location_code', 'position_str', 'x', 'y', 'z', 'zone', 'aisle', 'slot', 'dist_from_origin']]

# Hợp nhất toàn bộ vị trí xuất hiện trong Picking_Wave và 4 file chiến lược kho
all_source_locs = (
    set(raw_wave['locations'].str.strip()) |
    set(raw_random['originalLocation'].str.strip()) |
    set(raw_classbased['Location'].str.strip()) |
    set(raw_dedicated['Location'].str.strip()) |
    set(raw_hybrid['Location'].str.strip())
)
known_locs = set(df_stg_location['location_code'])
extra_locs = sorted(list(all_source_locs - known_locs))

if extra_locs:
    print(f"   Bổ sung {len(extra_locs)} vị trí từ Picking Wave và các layout kho...")
    extra_rows = []
    for sloc in extra_locs:
        parts = sloc.split('-')
        zone_prefix = parts[0]
        aisle_val = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
        slot_val  = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
        extra_rows.append({
            'location_code'   : sloc,
            'position_str'    : '0, 0, 1',
            'x'               : 0.0,
            'y'               : 0.0,
            'z'               : 1.0,
            'zone'            : zone_prefix,
            'aisle'           : aisle_val,
            'slot'            : slot_val,
            'dist_from_origin': 0.0
        })
    df_stg_location = pd.concat([df_stg_location, pd.DataFrame(extra_rows)], ignore_index=True)

# ── Bổ sung các trường phục vụ bài toán Decision Support System (DSS) ──
# 1. floor_type: Tầng Z <= 2 là Manual Picking (lấy hàng thuận tiện), Z > 2 là Replenishment (kệ cao/dự phòng)
df_stg_location['floor_type'] = df_stg_location['z'].apply(
    lambda z_val: 'Manual Picking' if z_val <= 2 else 'Replenishment'
)

# 2. xyz_code: Độ ưu tiên vị trí kho cho nhân viên lấy hàng (Dedicated Storage XYZCOD)
# X: Vị trí vàng gần hành lang chính, Y: Trung bình, Z: Sâu trong dãy, C: Dự phòng
loc_xyz_map = raw_dedicated.set_index(raw_dedicated['Location'].str.strip())['XYZCOD'].str.strip().to_dict()
df_stg_location['xyz_code'] = df_stg_location['location_code'].map(loc_xyz_map).fillna('C')

# 3. abc_zone: Phân loại khu vực theo tần suất xuất kho (Class-Based Storage ABCCOD)
loc_abc_map = raw_classbased.set_index(raw_classbased['Location'].str.strip())['ABCCOD'].str.strip().to_dict()
df_stg_location['abc_zone'] = df_stg_location['location_code'].map(loc_abc_map).fillna('C')

# 4. rack_capacity: Sức chứa tiêu chuẩn mỗi giá kệ là 18 ô (slot col_1 đến col_18)
df_stg_location['rack_capacity'] = 18

print(f"   Hoàn tất: {len(df_stg_location)} vị trí lưu trữ (Kèm xyz_code, abc_zone, floor_type, rack_capacity).")
display(df_stg_location.head(3))


🔄 [2/6] Làm sạch Storage_Location...
   Bổ sung 65 vị trí từ Picking Wave và các layout kho...
   Hoàn tất: 2357 vị trí lưu trữ.


,location_code,position_str,x,y,z,zone,aisle,slot,dist_from_origin
0,A-14-11,"368, 0, 1",368.0,0.0,1.0,A,14,11,368.14
1,A-14-12,"352, 0, 1",352.0,0.0,1.0,A,14,12,352.14
2,A-14-13,"336, 0, 1",336.0,0.0,1.0,A,14,13,336.15


In [5]:
# ── 2.3 Làm sạch Customer_Order ──────────────────────────────────
print("🔄 [3/6] Làm sạch Customer_Order...")
df_stg_order = raw_order.copy()

for col in ['codCustomer', 'Reference', 'operator', 'creationDate']:
    df_stg_order[col] = df_stg_order[col].astype(str).str.strip()

# Điền 16 giá trị thiếu ở Size (US) theo median của SKU
size_by_ref = df_stg_order.groupby('Reference')['Size (US)'].median()
global_size_median = df_stg_order['Size (US)'].median()
df_stg_order['Size (US)'] = df_stg_order['Size (US)'].fillna(df_stg_order['Reference'].map(size_by_ref)).fillna(global_size_median)

# Parse creationDate sang datetime chuẩn
df_stg_order['creation_datetime'] = pd.to_datetime(df_stg_order['creationDate'], format='%d/%m/%Y %H:%M')
df_stg_order['order_date']        = df_stg_order['creation_datetime'].dt.strftime('%Y-%m-%d')
df_stg_order['date_key']          = df_stg_order['creation_datetime'].dt.strftime('%Y%m%d').astype(int)

df_stg_order = df_stg_order.rename(columns={
    'codCustomer'     : 'customer_code',
    'orderNumber'     : 'order_number',
    'orderToCollect'  : 'order_to_collect',
    'Reference'       : 'reference',
    'Size (US)'       : 'size_us',
    'quantity (units)': 'quantity_units',
    'waveNumber'      : 'wave_number',
    'operator'        : 'operator_name'
})[['order_number', 'order_to_collect', 'customer_code', 'reference', 'size_us', 
    'quantity_units', 'creation_datetime', 'order_date', 'date_key', 'wave_number', 'operator_name']]

print(f"   Hoàn tất: {len(df_stg_order)} dòng đơn hàng. Missing size: {df_stg_order['size_us'].isnull().sum()}.")
display(df_stg_order.head(3))

🔄 [3/6] Làm sạch Customer_Order...
   Hoàn tất: 122370 dòng đơn hàng. Missing size: 0.


,order_number,order_to_collect,customer_code,reference,size_us,quantity_units,creation_datetime,order_date,date_key,wave_number,operator_name
0,124438,8,C0000016,8N10W9,9.0,6,2023-10-19 07:18:00,2023-10-19,20231019,43175,Operator_1
1,124438,7,C0000016,2T1DJM,13.0,1,2023-10-19 07:18:00,2023-10-19,20231019,43175,Operator_1
2,124438,6,C0000016,QVFB0Q,105.0,1,2023-10-19 07:18:00,2023-10-19,20231019,43175,Operator_1


In [6]:
# ── 2.4 Làm sạch Picking_Wave ────────────────────────────────────
print("🔄 [4/6] Làm sạch Picking_Wave...")
df_stg_picking = raw_wave.copy()

for col in ['reference', 'locations', 'operator']:
    df_stg_picking[col] = df_stg_picking[col].astype(str).str.strip()

# Gắn thông tin ngày của Wave từ Customer_Order
wave_date_map = df_stg_order.groupby('wave_number')['date_key'].min().to_dict()
df_stg_picking['date_key'] = df_stg_picking['waveNumber'].map(wave_date_map)

# Gắn tọa độ và cự ly từ df_stg_location
loc_info = df_stg_location.set_index('location_code')[['x', 'y', 'z', 'dist_from_origin']].to_dict('index')

df_stg_picking['x'] = df_stg_picking['locations'].map(lambda l: loc_info.get(l, {}).get('x', 0.0))
df_stg_picking['y'] = df_stg_picking['locations'].map(lambda l: loc_info.get(l, {}).get('y', 0.0))
df_stg_picking['z'] = df_stg_picking['locations'].map(lambda l: loc_info.get(l, {}).get('z', 1.0))
df_stg_picking['dist_from_origin'] = df_stg_picking['locations'].map(lambda l: loc_info.get(l, {}).get('dist_from_origin', 0.0))

df_stg_picking = df_stg_picking.rename(columns={
    'waveNumber'            : 'wave_number',
    'reference'             : 'reference',
    'Size (US)'             : 'size_us',
    'quantityToPick (units)': 'qty_to_pick',
    'locations'             : 'location_code',
    'operator'              : 'operator_name'
})[['wave_number', 'date_key', 'reference', 'size_us', 'qty_to_pick', 
    'location_code', 'operator_name', 'x', 'y', 'z', 'dist_from_origin']]

print(f"   Hoàn tất: {len(df_stg_picking)} tác vụ lấy hàng.")
display(df_stg_picking.head(3))

🔄 [4/6] Làm sạch Picking_Wave...
   Hoàn tất: 215192 tác vụ lấy hàng.


,wave_number,date_key,reference,size_us,qty_to_pick,location_code,operator_name,x,y,z,dist_from_origin
0,43175,20231019,8N10W9,9.0,1,H-06-13,Operator_1,534.0,688.0,1.0,870.98
1,43175,20231019,8N10W9,9.0,1,H-06-13,Operator_1,534.0,688.0,1.0,870.98
2,43175,20231019,8N10W9,9.0,1,H-06-13,Operator_1,534.0,688.0,1.0,870.98


In [7]:
# ── 2.5 Làm sạch Support_Points_Navigation ───────────────────────
print("🔄 [5/6] Làm sạch Support_Points_Navigation...")
df_stg_nav = raw_nav.copy()

coords = df_stg_nav['points_specified'].str.extract(r'\(([^,]+),\s*([^,]+),\s*([^)]+)\)').astype(float)
df_stg_nav['x_nav'] = coords[0]
df_stg_nav['y_nav'] = coords[1]
df_stg_nav['z_nav'] = coords[2]
df_stg_nav['label'] = df_stg_nav['labels'].astype(str).str.strip()

df_stg_nav = df_stg_nav[['label', 'x_nav', 'y_nav', 'z_nav']]
print(f"   Hoàn tất: {len(df_stg_nav)} điểm điều hướng.")
display(df_stg_nav.head(3))

🔄 [5/6] Làm sạch Support_Points_Navigation...
   Hoàn tất: 44 điểm điều hướng.


,label,x_nav,y_nav,z_nav
0,LC-01,66.0,-29.0,1.0
1,LC-02,66.0,61.0,1.0
2,LC-03,66.0,151.0,1.0


In [8]:
# ── 2.6 Unpivot 4 Chiến Lược Bố Trí Kho ──────────────────────────
print("🔄 [6/6] Unpivot 4 Chiến Lược Lưu Trữ...")

def melt_strategy(df_raw, loc_col, class_col, strategy_name):
    id_vars = [loc_col] + ([class_col] if class_col else [])
    val_vars = [c for c in df_raw.columns if c not in id_vars]
    
    melted = df_raw.melt(id_vars=id_vars, value_vars=val_vars, var_name='slot_raw', value_name='slot_val')
    melted['strategy'] = strategy_name
    melted['location_code'] = melted[loc_col].astype(str).str.strip()
    
    if class_col:
        melted['classification_code'] = melted[class_col].astype(str).str.strip().str.upper()
    else:
        melted['classification_code'] = None
        
    melted['slot_number'] = melted['slot_raw'].astype(str).str.replace('col_', '', regex=False).astype(int)
    
    parts = melted['slot_val'].astype(str).str.split(';', expand=True)
    melted['reference'] = parts[0].str.strip()
    melted['size_us']   = pd.to_numeric(parts[1], errors='coerce')
    
    return melted[['strategy', 'location_code', 'classification_code', 'slot_number', 'reference', 'size_us']]

m_rnd = melt_strategy(raw_random,     'originalLocation', None,     'Random')
m_cbs = melt_strategy(raw_classbased, 'Location',         'ABCCOD', 'ClassBased')
m_ded = melt_strategy(raw_dedicated,  'Location',         'XYZCOD', 'Dedicated')
m_hyb = melt_strategy(raw_hybrid,     'Location',         'XYZCOD', 'Hybrid')

df_stg_storage = pd.concat([m_rnd, m_cbs, m_ded, m_hyb], ignore_index=True)

print(f"   Hoàn tất unpivot: {len(df_stg_storage):,} records phân bổ ô kệ.")
display(df_stg_storage.head(4))

🔄 [6/6] Unpivot 4 Chiến Lược Lưu Trữ...
   Hoàn tất unpivot: 167,634 records phân bổ ô kệ.


,strategy,location_code,classification_code,slot_number,reference,quantity
0,Random,A-14-11,None,1,TQBVRI,7.0
1,Random,A-14-12,None,1,YE890X,5.0
2,Random,A-14-13,None,1,M8W7TS,12.0
3,Random,A-14-21,None,1,CST1KC,5.0


---
## 3. LOAD TO STAGING — Xuất Dữ Liệu Sạch Ra `data/staging/`

In [9]:
print("💾 Đang xuất các bảng sạch ra data/staging/...")

files_to_export = [
    ('stg_products.csv',   df_stg_product),
    ('stg_locations.csv',  df_stg_location),
    ('stg_orders.csv',     df_stg_order),
    ('stg_picking.csv',    df_stg_picking),
    ('stg_nav_points.csv', df_stg_nav),
    ('stg_storage.csv',    df_stg_storage),
]

summary_records = []

for filename, df in files_to_export:
    out_path = STG_DIR / filename
    df.to_csv(out_path, index=False, encoding='utf-8')
    size_mb = out_path.stat().st_size / (1024 * 1024)
    summary_records.append({
        'File Staging': filename,
        'Số dòng': f"{len(df):,}",
        'Số cột': len(df.columns),
        'Dung lượng': f"{size_mb:.2f} MB",
        'Null values': df.isnull().sum().sum()
    })

print("=" * 70)
print("✅ TẤT CẢ FILE STAGING ĐÃ ĐƯỢC TẠO THÀNH CÔNG!")
print("=" * 70)
display(pd.DataFrame(summary_records))

💾 Đang xuất các bảng sạch ra data/staging/...
✅ TẤT CẢ FILE STAGING ĐÃ ĐƯỢC TẠO THÀNH CÔNG!


,File Staging,Số dòng,Số cột,Dung lượng,Null values
0,stg_products.csv,317,3,0.00 MB,0
1,stg_locations.csv,"2,357",9,0.12 MB,0
2,stg_orders.csv,"122,370",11,10.54 MB,0
3,stg_picking.csv,"215,192",11,14.70 MB,0
4,stg_nav_points.csv,44,4,0.00 MB,0
5,stg_storage.csv,"167,634",6,5.36 MB,41256
